## Initialization

In [ ]:
#Imports
import numpy as np
import torch
from torch import nn
import torchaudio
from torch.utils.data import DataLoader
from torchvision import transforms

#Custom modules
import cAudiotools
import cLogger
import cTransforms
import cModelManager

MODEL_NAME = "CNN_Raw"
model_description = "CNN for VAD regression in speech using only raw audio as input."

#Define output paths
log = cLogger.Log("output\logs", prefix=MODEL_NAME)
model_dir = cModelManager.Directories.make_unique(f"output\models\{MODEL_NAME}")
log.log_property("model_name", MODEL_NAME)
log.log_property("model_description", model_description)
log.log_property("model_dir", model_dir)

#Torch properties and device
log.log_property("torch_version", torch.__version__, show=True)
log.log_property("torchaudio_version", torchaudio.__version__, show=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    log.log_property("device", "cuda", show=True)
    log.log_property("GPU_count", torch.cuda.device_count(), show=True)
    log.log_property("GPU_device", torch.cuda.get_device_name(0), show=True)
else:
    log.log_property("device", "cpu", show=True)
    

## Define data parameters

In [2]:
#Parameters:
loader_params = {
    "dataset_train": r"C:\Datasets\_compiled\msp-podcast-2\labels_train_VAD.csv",
    "dataset_train_dir": r"C:\Datasets\_compiled\msp-podcast-2\Train",
    "dataset_dev": r"C:\Datasets\_compiled\msp-podcast-2\labels_development_VAD.csv",
    "dataset_dev_dir": r"C:\Datasets\_compiled\msp-podcast-2\Development",
    "batch_size": 16,
    "shuffle": True,
    "collate_function": cAudiotools.Batching.waveform_dynamic,
    "target_transform": cTransforms.NormalizeMinus(1, 7)
}

training_params = {
    "learning_rate": 1e-3,
    "epochs": 200,
    "batch_size": 16,
    "adam_betas": (0.9, 0.999),
    "adam_epsilon": 1e-8
}

audio_params = {
    "sample_rate": 16000
}

spectrogram_params = {
    "n_fft": 512,
    "hop_length": 160,
    "win_length": 400,
    "window_fn": torch.hamming_window,
    "normalized": True,
    "power": 2
}

mel_params = {
    "n_mels": 64,
    "pad_mode": "constant",
    "mel_scale": "htk",
    "n_mfcc": 20
}


## Define training parameters

In [3]:

spectogram_transform = torchaudio.transforms.Spectrogram(
    n_fft=spectrogram_params["n_fft"],
    hop_length=spectrogram_params["hop_length"],
    win_length=spectrogram_params["win_length"],
    window_fn=spectrogram_params["window_fn"],
    normalized=spectrogram_params["normalized"],
    power=spectrogram_params["power"]
    )

melspectogram_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=audio_params["sample_rate"],
    n_fft=spectrogram_params["n_fft"],
    win_length=spectrogram_params["win_length"],
    hop_length=spectrogram_params["hop_length"],
    window_fn=spectrogram_params["window_fn"],
    power=spectrogram_params["power"],
    n_mels=mel_params["n_mels"],
    normalized=spectrogram_params["normalized"],
    pad_mode=mel_params["pad_mode"],
    mel_scale=mel_params["mel_scale"]
)

mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=audio_params["sample_rate"],
    n_mfcc=mel_params["n_mfcc"],
    melkwargs={
        "n_fft": spectrogram_params["n_fft"],
        "win_length": spectrogram_params["win_length"],
        "hop_length": spectrogram_params["hop_length"],
        "window_fn": spectrogram_params["window_fn"],
        "power": spectrogram_params["power"],
        "n_mels": mel_params["n_mels"],
        "normalized": spectrogram_params["normalized"],
        "pad_mode": mel_params["pad_mode"],
        "mel_scale": mel_params["mel_scale"]
    }
)
loader_params["data_transform"] = None


log.log_properties("Loader", loader_params)
log.log_properties("Training", training_params)
log.log_properties("Audio", audio_params)
#log.log_properties("Spectrogram", spectrogram_params)
#log.log_properties("Mel Spectrogram", mel_params)

#Train set
msp_vad_train = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_train"], 
    loader_params["dataset_train_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
train_dataloader = DataLoader(
    msp_vad_train,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )

#Development (validation) set
msp_vad_dev = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_dev"],
    loader_params["dataset_dev_dir"],
    transform=spectogram_transform,
    target_transform=loader_params["target_transform"]
    )
dev_dataloader = DataLoader(
    msp_vad_dev,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )


## Define Architecture

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.stack = nn.Sequential(
            nn.Conv1d(1, 1, kernel_size=7, stride=1, padding='valid'),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Linear(1, 12),
            nn.ReLU(),
            nn.Linear(12, 3)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.stack(x)
        return logits

model = NeuralNetwork().to(device)
log.log_property("model_structure", str(model), show=True)

loss_fn = nn.MSELoss()
log.log_property("loss_function", str(loss_fn), show=True)

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=training_params["learning_rate"], 
    betas=training_params["adam_betas"],
    eps=training_params["adam_epsilon"]
    )

log.log_property("optimizer", str(optimizer), show=True)

## Training

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (inputs, outputs) in enumerate(dataloader):
        inputs, outputs = inputs.to(device), outputs.to(device)

        pred = model(inputs)
        loss = loss_fn(pred, outputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(inputs)
            log.log_message(f"train loss: {loss:>7f}  [{current:>5d}/{size:>5d}]", show=True)

def validation_loop(dataloader, model, loss_fn, epoch=None):
    model.eval()
    num_batches = len(dataloader.dataset)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()

    test_loss /= num_batches
    if epoch:
        log.log_epoch(epoch, {"dev_avg_loss": test_loss}, show=True)
    else:
        log.log_message(f"Avg loss: {test_loss:>8f} \n", show=True)

log.track_time(True, message="Starting training.")
for epoch in range(training_params["epochs"]):
    train_loop(train_dataloader, model, loss_fn, optimizer)
    validation_loop(dev_dataloader, model, loss_fn, epoch=epoch)
    log.log_elapsed_time(message=f"Epoch {epoch+1} completed", show=True)
    log.save()
log.log_elapsed_time(message="Training completed", show=True)
track_time=False
log.save_txt()